In [8]:
import polars as pl
import os

from pathlib import Path

In [9]:
WORKING_PATH = Path('/group/pmc021/amunif/epi-thesis/workflow/18_Pairwise Ranking HepG2 REMC/')
DATASET_PATH = WORKING_PATH / 'dataset'
OUTPUT_PATH  = WORKING_PATH / 'output' / 'combined'

In [10]:
### Merge the results into single CSV file

# Read all CSV into single dataframe
pl_df = pl.read_csv(OUTPUT_PATH / 'test' / "*-test-metrics.csv")

In [11]:
pl_df

model,seed,histone_marker,epochs_trained,val_accuracy,val_auc,test_accuracy,test_auc,test_precision,test_recall,test_f1,antisymmetry
str,i64,str,i64,f64,f64,f64,f64,f64,f64,f64,f64
"""DirectRanker""",1011,"""H3K4me3""",12,75.4,0.84,74.9,0.8437,0.7218,0.7745,0.7472,0.922
"""LogisticRegression""",1011,"""H3K4me3""",null,75.2,0.8226,76.6,0.8425,0.7966,0.6868,0.7377,0.797
"""RandomForest""",1011,"""H3K4me3""",null,75.9,0.8281,77.2,0.8531,0.7898,0.714,0.75,0.788
"""SVM_Linear""",1011,"""H3K4me3""",null,75.1,0.8231,76.8,0.8428,0.7976,0.691,0.7405,0.794
"""DirectRanker""",123,"""H3K4me3""",14,77.1,0.86,73.0,0.8044,0.7097,0.7857,0.7458,0.922
…,…,…,…,…,…,…,…,…,…,…,…
"""SVM_Linear""",456,"""H3K4me3-H3K9ac-H3K9me3-H3K27ac…",null,75.2,0.8224,75.9,0.8204,0.7665,0.7337,0.7497,0.759
"""DirectRanker""",789,"""H3K4me3-H3K9ac-H3K9me3-H3K27ac…",54,79.2,0.87,76.4,0.8592,0.7436,0.7835,0.7631,0.989
"""LogisticRegression""",789,"""H3K4me3-H3K9ac-H3K9me3-H3K27ac…",null,73.4,0.8155,72.9,0.8125,0.7238,0.7134,0.7186,0.777


In [12]:
summary_df = (
    pl_df
    .group_by(["histone_marker", "model"])
    .agg([
        pl.col("val_accuracy").mean().alias("val_accuracy_mean"),
        pl.col("val_accuracy").std().alias("val_accuracy_std"),
        pl.col("val_auc").mean().alias("val_auc_mean"),
        pl.col("val_auc").std().alias("val_auc_std"),
        pl.col("test_accuracy").mean().alias("test_accuracy_mean"),
        pl.col("test_accuracy").std().alias("test_accuracy_std"),
        pl.col("test_auc").mean().alias("test_auc_mean"),
        pl.col("test_auc").std().alias("test_auc_std"),
    ])
    .sort(["histone_marker", "test_accuracy_mean"], descending=[False, True])
    .with_columns(pl.col(pl.Float64).round(4))
)

In [13]:
summary_df

histone_marker,model,val_accuracy_mean,val_accuracy_std,val_auc_mean,val_auc_std,test_accuracy_mean,test_accuracy_std,test_auc_mean,test_auc_std
str,str,f64,f64,f64,f64,f64,f64,f64,f64
"""H3K27ac""","""RandomForest""",77.06,1.1739,0.8489,0.0144,75.74,1.4673,0.8446,0.0143
"""H3K27ac""","""LogisticRegression""",75.16,1.1696,0.8109,0.0157,73.18,1.3161,0.7987,0.0141
"""H3K27ac""","""SVM_Linear""",75.04,1.417,0.8101,0.0158,73.18,1.4464,0.8009,0.0134
"""H3K27ac""","""DirectRanker""",73.1,2.1378,0.834,0.0152,71.96,1.6965,0.8254,0.014
"""H3K27ac-H3K27me3""","""RandomForest""",77.14,1.6772,0.8497,0.015,74.38,1.1345,0.8412,0.0121
…,…,…,…,…,…,…,…,…,…
"""H3K9me3-H3K27ac-H3K27me3""","""SVM_Linear""",71.62,1.3755,0.7786,0.0243,70.68,1.7922,0.7751,0.0179
"""H3K9me3-H3K27me3""","""RandomForest""",67.38,1.3664,0.734,0.0211,65.38,2.3339,0.7163,0.026
"""H3K9me3-H3K27me3""","""DirectRanker""",66.0,1.4422,0.726,0.023,65.06,2.2188,0.7139,0.0242


In [14]:
summary_df.write_csv(OUTPUT_PATH/ f"HepG2 REMC E118 Ranking.csv", include_header=True)